In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp05
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


# ── shared/mlfp05/ex_5.py ──
"""
Shared utilities for Exercise 5 — GANs and Generative Models.

Infrastructure only: data loading, visualisation helpers, metric computation,
and kailash-ml engine setup. No domain logic or business scenarios.
"""

import asyncio
import pickle
from pathlib import Path
from typing import TYPE_CHECKING

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import torchvision
import matplotlib.pyplot as plt
from matplotlib.figure import Figure

from kailash.db import ConnectionManager
from kailash_ml import ExperimentTracker
from kailash_ml import ModelRegistry
from kailash_ml.types import MetricSpec


if TYPE_CHECKING:
    from kailash_ml.engines.model_registry import ModelVersion

# ════════════════════════════════════════════════════════════════════════
# Constants
# ════════════════════════════════════════════════════════════════════════
LATENT_DIM = 64
IMG_DIM = 28 * 28
BATCH_SIZE = 128
try:
    _HERE = Path(__file__).resolve()
    REPO_ROOT = _HERE.parents[2]
except NameError:
    REPO_ROOT = Path.cwd()
DATA_DIR = REPO_ROOT / "data" / "mlfp05" / "mnist"
OUTPUT_DIR = Path("outputs") / "ex5_gans"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ════════════════════════════════════════════════════════════════════════
# Environment and device
# ════════════════════════════════════════════════════════════════════════
def init_environment() -> torch.device:
    """Set up environment, seeds, and return the compute device."""
    setup_environment()
    torch.manual_seed(42)
    np.random.seed(42)
    device = get_device()
    print(f"Using device: {device}")
    return device


# ════════════════════════════════════════════════════════════════════════
# Data loading
# ════════════════════════════════════════════════════════════════════════
def load_mnist(device: torch.device) -> tuple[torch.Tensor, torch.Tensor, DataLoader]:
    """Load full MNIST (60K), scale to [-1, 1] for tanh generators.

    Returns:
        X_real: (60000, 1, 28, 28) tensor on device, range [-1, 1]
        y_real: (60000,) long tensor on device
        real_loader: DataLoader with batch_size=128, shuffle, drop_last
    """
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    train_set = torchvision.datasets.MNIST(
        root=str(DATA_DIR),
        train=True,
        download=True,
        transform=torchvision.transforms.ToTensor(),
    )

    X_real = torch.stack([train_set[i][0] for i in range(len(train_set))])
    y_real = torch.tensor(
        [train_set[i][1] for i in range(len(train_set))], dtype=torch.long
    )
    X_real = (X_real * 2.0 - 1.0).to(device)
    y_real = y_real.to(device)

    print(
        f"MNIST: {len(X_real)} digits, shape {tuple(X_real.shape[1:])}, "
        f"pixel range [{X_real.min():.2f}, {X_real.max():.2f}]"
    )
    class_dist = ", ".join(f"{c}={int((y_real == c).sum())}" for c in range(10))
    print(f"  class distribution: {class_dist}")

    real_loader = DataLoader(
        TensorDataset(X_real), batch_size=BATCH_SIZE, shuffle=True, drop_last=True
    )
    return X_real, y_real, real_loader


def load_mnist_test(device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """Load the 10K MNIST test digits — images no generator was trained on.

    Returns:
        X_test: (10000, 1, 28, 28) tensor on device, range [-1, 1]
        y_test: (10000,) long tensor on device
    """
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    test_set = torchvision.datasets.MNIST(
        root=str(DATA_DIR),
        train=False,
        download=True,
        transform=torchvision.transforms.ToTensor(),
    )
    X_test = torch.stack([test_set[i][0] for i in range(len(test_set))])
    y_test = torch.tensor(
        [test_set[i][1] for i in range(len(test_set))], dtype=torch.long
    )
    return (X_test * 2.0 - 1.0).to(device), y_test.to(device)


# ════════════════════════════════════════════════════════════════════════
# Kailash engine setup
# ════════════════════════════════════════════════════════════════════════
def setup_engines() -> (
    tuple[ConnectionManager, ExperimentTracker, str, ModelRegistry | None]
):
    """Create ExperimentTracker (kailash-ml 1.1.1 factory) and ModelRegistry.

    Returns:
        conn, tracker, experiment_name, registry
    """

    async def _setup():
        # Schema-conflict workaround (kailash-ml 1.5.x): ExperimentTracker
        # and ModelRegistry use incompatible _kml_model_versions schemas.
        # Route them to separate sqlite files until upstream fixes the conflict.
        db = "sqlite:///mlfp05_gans.db"
        registry_db = "sqlite:///mlfp05_gans_registry.db"
        tracker = await ExperimentTracker.create(store_url=db)
        conn = ConnectionManager(registry_db)
        await conn.initialize()
        registry = ModelRegistry(conn)
        return conn, tracker, "m5_gans", registry

    return asyncio.run(_setup())


async def close_engines(conn: ConnectionManager) -> None:
    """Cleanly shut down the connection manager."""
    await conn.close()


# ════════════════════════════════════════════════════════════════════════
# Generator and Discriminator architectures
# ════════════════════════════════════════════════════════════════════════
class Generator(nn.Module):
    """MLP Generator: z -> 784-d -> (1, 28, 28).

    A fully-connected (MLP) generator, not a convolutional DCGAN. It borrows
    the DCGAN training recipe — BatchNorm + LeakyReLU in hidden layers and a
    Tanh output to match the [-1, 1] image scaling. Linear layers that feed
    BatchNorm have no bias: BatchNorm subtracts the batch mean, so such a
    bias would be cancelled and would receive exactly zero gradient (which
    the Prescription Pad would report as a vanishing-gradient layer).
    """

    def __init__(self, latent_dim: int = LATENT_DIM, hidden: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, hidden, bias=False),
            nn.BatchNorm1d(hidden),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden, hidden * 2, bias=False),
            nn.BatchNorm1d(hidden * 2),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden * 2, IMG_DIM),
            nn.Tanh(),
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.net(z).view(-1, 1, 28, 28)


class Discriminator(nn.Module):
    """MLP Discriminator: 28x28 -> scalar logit.

    Dropout prevents D from overfitting to real images (memorising
    instead of learning distributional features).
    """

    def __init__(self, hidden: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(IMG_DIM, hidden * 2),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(hidden * 2, hidden),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(hidden, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


# ════════════════════════════════════════════════════════════════════════
# FID feature extractor
# ════════════════════════════════════════════════════════════════════════
class LeNetFeatureExtractor(nn.Module):
    """CNN feature extractor for FID computation.

    Returns 64-dim feature vectors (analogous to InceptionV3 pool3 layer).
    We use a domain-specific extractor because InceptionV3 expects 299x299 RGB;
    for 28x28 grayscale MNIST a trained LeNet gives more meaningful distances.
    """

    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 5, stride=2, padding=2),
            nn.ReLU(),
            nn.Conv2d(16, 32, 5, stride=2, padding=2),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 64),
            nn.ReLU(),
        )
        self.classifier = nn.Linear(64, 10)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.features(x))

    def extract_features(self, x: torch.Tensor) -> torch.Tensor:
        return self.features(x)


def train_feature_extractor(
    X_real: torch.Tensor,
    y_real: torch.Tensor,
    device: torch.device,
    epochs: int = 5,
) -> LeNetFeatureExtractor:
    """Train the LeNet feature extractor on MNIST for FID computation.

    Returns the trained extractor in eval mode.
    """
    print("\n== Training feature extractor (for FID + mode coverage) ==")
    extractor = LeNetFeatureExtractor().to(device)
    opt = torch.optim.Adam(extractor.parameters(), lr=1e-3)
    X_01 = (X_real + 1.0) / 2.0  # [0, 1] for classifier

    for epoch in range(epochs):
        losses = []
        for xb, yb in DataLoader(
            TensorDataset(X_01, y_real), batch_size=256, shuffle=True
        ):
            loss = F.cross_entropy(extractor(xb), yb)
            opt.zero_grad()
            loss.backward()
            opt.step()
            losses.append(loss.item())
        with torch.no_grad():
            acc = (extractor(X_01[:10000]).argmax(-1) == y_real[:10000]).float().mean()
        print(f"  epoch {epoch+1}/{epochs}  loss={np.mean(losses):.3f}  acc={acc:.3f}")

    extractor.eval()
    return extractor


# ════════════════════════════════════════════════════════════════════════
# FID computation
# ════════════════════════════════════════════════════════════════════════
def compute_fid(
    extractor: LeNetFeatureExtractor,
    real: torch.Tensor,
    generated: torch.Tensor,
) -> float:
    """Frechet distance between real and generated feature distributions.

    FID = ||mu_r - mu_g||^2 + Tr(Sigma_r + Sigma_g - 2*sqrt(Sigma_r @ Sigma_g))

    Computed in the 64-d LeNet feature space (not Inception), so values are
    only comparable with other values from the same extractor. Lower = the
    generated distribution is closer to the real one (fidelity AND
    diversity). Uses eigenvalues of Sigma_r @ Sigma_g (no scipy dependency).
    """
    extractor.eval()

    def _extract(images: torch.Tensor) -> np.ndarray:
        feats = []
        with torch.no_grad():
            for i in range(0, len(images), 512):
                feats.append(
                    extractor.extract_features(images[i : i + 512]).cpu().numpy()
                )
        return np.concatenate(feats)

    rf, gf = _extract(real), _extract(generated)
    mu_r, mu_g = rf.mean(0), gf.mean(0)
    sig_r, sig_g = np.cov(rf, rowvar=False), np.cov(gf, rowvar=False)

    diff = mu_r - mu_g
    # Sigma_r @ Sigma_g is a product of two symmetric PSD matrices — it is
    # NOT itself symmetric, so np.linalg.eigh (Hermitian-only) reads the wrong
    # triangle and returns garbage (this produced large-negative FID). Its
    # eigenvalues are nonetheless real and non-negative (it is similar to the
    # symmetric PSD matrix Sigma_r^½ Sigma_g Sigma_r^½), so use the general
    # eigensolver and the identity Tr(sqrt(AB)) = Σ sqrt(λ_i(AB)).
    eigvals = np.linalg.eigvals(sig_r @ sig_g)
    eigvals = np.maximum(np.real(eigvals), 0.0)
    trace_sqrt = float(np.sum(np.sqrt(eigvals)))

    return float(diff @ diff + np.trace(sig_r) + np.trace(sig_g) - 2 * trace_sqrt)


# ════════════════════════════════════════════════════════════════════════
# Mode coverage diagnostic
# ════════════════════════════════════════════════════════════════════════
def mode_coverage(
    G: Generator,
    classifier: LeNetFeatureExtractor,
    device: torch.device,
    n: int = 5000,
) -> tuple[int, dict[int, int], float]:
    """Measure mode coverage: how many of the 10 digit classes the generator produces.

    Returns:
        n_classes: number of unique classes generated
        per_class_counts: dict mapping class -> count
        shannon_entropy: diversity measure (max = log2(10) = 3.32 for uniform)
    """
    G.eval()
    classifier.eval()
    with torch.no_grad():
        fake_01 = (G(torch.randn(n, LATENT_DIM, device=device)) + 1) / 2
        preds = classifier(fake_01).argmax(-1).cpu().numpy()
    unique, counts = np.unique(preds, return_counts=True)
    probs = counts / counts.sum()
    entropy = float(-np.sum(probs * np.log2(probs + 1e-10)))
    return int(len(unique)), {int(k): int(v) for k, v in zip(unique, counts)}, entropy


# ════════════════════════════════════════════════════════════════════════
# Visualisation helpers
# ════════════════════════════════════════════════════════════════════════
def plot_image_grid(
    images: torch.Tensor,
    nrow: int = 8,
    ncol: int = 8,
    title: str = "Generated Images",
    save_path: str | None = None,
) -> Figure:
    """Plot an 8x8 grid of generated images.

    Args:
        images: (N, 1, 28, 28) tensor in [-1, 1] range
        nrow, ncol: grid dimensions
        title: figure title
        save_path: optional path to save the figure
    """
    n = min(nrow * ncol, len(images))
    fig, axes = plt.subplots(nrow, ncol, figsize=(12, 12))
    fig.suptitle(title, fontsize=16, fontweight="bold")

    for i in range(nrow * ncol):
        ax = axes[i // ncol][i % ncol]
        if i < n:
            img = images[i].detach().cpu().squeeze().numpy()
            img = (img + 1) / 2  # [-1,1] -> [0,1]
            ax.imshow(img, cmap="gray", vmin=0, vmax=1)
        ax.axis("off")

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"  Saved: {save_path}")
    return fig


def plot_latent_interpolation(
    G: Generator,
    device: torch.device,
    n_steps: int = 10,
    n_rows: int = 5,
    title: str = "Latent Space Interpolation",
    save_path: str | None = None,
) -> Figure:
    """Interpolate between pairs of random latent vectors.

    Shows smooth transitions between generated images — evidence that
    the generator has learned a continuous manifold, not memorised digits.
    """
    G.eval()
    fig, axes = plt.subplots(n_rows, n_steps, figsize=(n_steps * 1.2, n_rows * 1.4))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    with torch.no_grad():
        for row in range(n_rows):
            z1 = torch.randn(1, LATENT_DIM, device=device)
            z2 = torch.randn(1, LATENT_DIM, device=device)
            for col in range(n_steps):
                alpha = col / (n_steps - 1)
                z = (1 - alpha) * z1 + alpha * z2
                img = G(z).squeeze().cpu().numpy()
                img = (img + 1) / 2
                axes[row][col].imshow(img, cmap="gray", vmin=0, vmax=1)
                axes[row][col].axis("off")

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"  Saved: {save_path}")
    return fig


def plot_training_progression(
    G: Generator,
    device: torch.device,
    epoch_snapshots: dict[int, dict],
    title: str = "Training Progression",
    save_path: str | None = None,
) -> Figure:
    """Plot generated images at different training epochs.

    Shows how generation quality improves over training — from random
    noise to recognisable digits.

    Args:
        epoch_snapshots: dict of {epoch: state_dict} captured during training
    """
    n_snapshots = len(epoch_snapshots)
    n_samples = 8
    fig, axes = plt.subplots(
        n_snapshots, n_samples, figsize=(n_samples * 1.4, n_snapshots * 1.6)
    )
    fig.suptitle(title, fontsize=14, fontweight="bold")

    fixed_z = torch.randn(n_samples, LATENT_DIM, device=device)

    for row, (epoch, state_dict) in enumerate(sorted(epoch_snapshots.items())):
        G.load_state_dict(state_dict)
        G.eval()
        with torch.no_grad():
            imgs = G(fixed_z)
        for col in range(n_samples):
            ax = axes[row][col] if n_snapshots > 1 else axes[col]
            img = imgs[col].squeeze().cpu().numpy()
            img = (img + 1) / 2
            ax.imshow(img, cmap="gray", vmin=0, vmax=1)
            ax.axis("off")
            if col == 0:
                ax.set_ylabel(f"Epoch {epoch}", fontsize=10, rotation=0, labelpad=50)

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"  Saved: {save_path}")
    return fig


def plot_loss_curves(
    g_losses: list[float],
    d_losses: list[float],
    title: str = "Training Dynamics",
    g_label: str = "Generator",
    d_label: str = "Discriminator",
    save_path: str | None = None,
) -> Figure:
    """Plot G vs D loss curves across epochs."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    epochs = range(1, len(g_losses) + 1)

    # Individual losses
    ax1.plot(epochs, g_losses, "b-", linewidth=2, label=g_label)
    ax1.plot(epochs, d_losses, "r-", linewidth=2, label=d_label)
    ax1.set_xlabel("Epoch", fontsize=12)
    ax1.set_ylabel("Loss", fontsize=12)
    ax1.set_title("G vs D Loss", fontsize=13)
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)

    # G/D ratio — healthy GAN has ratio near 1
    ratio = [g / (d + 1e-8) for g, d in zip(g_losses, d_losses)]
    ax2.plot(epochs, ratio, "g-", linewidth=2)
    ax2.axhline(y=1.0, color="gray", linestyle="--", alpha=0.5, label="Balanced (1.0)")
    ax2.set_xlabel("Epoch", fontsize=12)
    ax2.set_ylabel("G/D Loss Ratio", fontsize=12)
    ax2.set_title("Training Balance", fontsize=13)
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"  Saved: {save_path}")
    return fig


# ════════════════════════════════════════════════════════════════════════
# Model registration helper
# ════════════════════════════════════════════════════════════════════════
def register_generator(
    registry: ModelRegistry | None,
    name: str,
    model: Generator,
    fid: float,
    coverage: int,
    entropy: float,
) -> "ModelVersion | None":
    """Register a trained generator in ModelRegistry with quality metrics."""
    if registry is None:
        print(f"  ModelRegistry not available — skipping {name}")
        return None

    async def _register():
        ver = await registry.register_model(
            name=f"m5_{name}",
            artifact=pickle.dumps(model.state_dict()),
            metrics=[
                MetricSpec(name="fid_score", value=fid),
                MetricSpec(name="mode_coverage", value=float(coverage)),
                MetricSpec(name="class_entropy", value=entropy),
            ],
        )
        print(
            f"  Registered {name}: v={ver.version}, FID={fid:.2f}, "
            f"coverage={coverage}/10"
        )
        return ver

    return asyncio.run(_register())




# ════════════════════════════════════════════════════════════════════════
# MLFP05 — Exercise 5.3: GAN Evaluation and Model Registry
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Why "looks good" is not a valid evaluation metric for GANs
#   - FID (Frechet Inception Distance) — the standard automated metric,
#     and why its numbers depend on the feature extractor
#   - Mode coverage analysis — detecting hidden mode collapse
#   - Shannon entropy as a diversity measure
#   - A nearest-neighbour novelty check for memorised training images
#   - Register trained generators in ModelRegistry with quality metrics
#   - Build a quality assurance pipeline for synthetic data production
#   - Apply: QA validation for the insurance company's synthetic data
#     pipeline before deploying to production fraud detection models
#
# PREREQUISITES: ex_5/01_vanilla_gan.py, ex_5/02_wgan_gp.py
# ESTIMATED TIME: ~40 min
# DATASET: MNIST — 60,000 real 28x28 grayscale digits
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import os

import numpy as np
import torch
import matplotlib.pyplot as plt

from kailash_ml import ModelVisualizer



## PHASE 1 — THEORY: Why GAN Evaluation Is Hard


THE PROBLEM:
  Unlike classifiers (accuracy, F1) or regressors (MSE, R²), GANs have
  no single ground truth to compare against. A generator that produces
  perfect images of the digit "7" — and ONLY "7" — has flawless
  individual images but is useless for any practical application.

  "LOOKS GOOD" IS DANGEROUS:
  Human visual inspection doesn't scale, is subjective, and misses
  subtle distribution mismatches. A GAN producing sharp but repetitive
  images fools the eye while silently failing on diversity.

  THREE EVALUATION DIMENSIONS:

  1. DISTRIBUTION MATCH (fidelity + diversity together):
     Does the cloud of generated images look like the cloud of real
     images? FID compares the two DISTRIBUTIONS — it is not a
     per-image score. Metric: FID (lower = better)

  2. DIVERSITY (mode coverage):
     Does the generator cover the full range of real data?
     Metric: mode coverage count + Shannon entropy

  3. NOVELTY (not memorising):
     Is the generator creating NEW images, not copying training data?
     Metric: nearest-neighbour distance to the training set, compared
     with the same distance for real images it never saw

  FID — FRECHET INCEPTION DISTANCE:

  FID treats both real and generated images as points in a learned
  feature space (from a pre-trained classifier). It then fits a
  multivariate Gaussian to each set and measures the distance between
  the two Gaussians:

    FID = ||mu_r - mu_g||^2 + Tr(Sig_r + Sig_g - 2*sqrt(Sig_r @ Sig_g))

  Intuition for professionals:
  - FID = 0: the two feature distributions are identical
  - FID has NO universal scale: it depends on the feature extractor.
    Published thresholds ("FID < 10 is excellent") are for 2048-d
    Inception features on natural images. We use a 64-d LeNet trained
    on MNIST (Inception expects 299x299 colour images), so our numbers
    are only comparable with other numbers from THIS extractor.
  - To make a number meaningful, compare it with a FLOOR: the FID
    between two sets of REAL digits. No generator can beat sampling
    noise, so "how many times the floor" is the readable figure.

  MODE COLLAPSE DETECTION:

  Mode collapse is the GAN equivalent of a factory that produces only
  one product. Shannon entropy quantifies diversity:
  - max entropy = log2(10) = 3.32 (uniform across all 10 digit classes)
  - entropy = 0: generator produces only one class (total collapse)

  FID does penalise missing modes, but it squeezes fidelity and
  diversity into ONE number: a middling FID cannot tell you whether
  every digit is slightly blurry or three digits are missing entirely.
  Coverage and entropy answer that second question directly.


In [ ]:
print("=" * 70)
print("  PHASE 1 — THEORY: The GAN Evaluation Problem")
print("=" * 70)
print(
)



## PHASE 2 — BUILD: FID Computation Pipeline + Feature Extractor


In [ ]:
print("\n" + "=" * 70)
print("  PHASE 2 — BUILD: FID Pipeline and Feature Extractor")
print("=" * 70)

device = init_environment()
X_real, y_real, real_loader = load_mnist(device)
conn, tracker, exp_name, registry = setup_engines()

# Train the feature extractor for FID computation
fid_ext = train_feature_extractor(X_real, y_real, device, epochs=5)



### Checkpoint 1


In [ ]:
fid_ext.eval()
with torch.no_grad():
    _test_acc = (
        (fid_ext((X_real[:1000] + 1) / 2).argmax(-1) == y_real[:1000]).float().mean()
    )
assert _test_acc > 0.8, f"Feature extractor accuracy too low: {_test_acc:.3f}"
print(f"  Feature extractor test accuracy: {_test_acc:.1%}")
print("\n--- Checkpoint 1 passed --- feature extractor trained\n")



## PHASE 3 — TRAIN: Both GAN Variants for Comparative Evaluation


In [ ]:
print("\n" + "=" * 70)
print("  PHASE 3 — TRAIN: Vanilla GAN + WGAN-GP for Comparison")
print("=" * 70)

import torch.nn as nn


# Train vanilla GAN
print("\n  Training Vanilla GAN (15 epochs)...")
G_gan = Generator().to(device)
D_gan = Discriminator().to(device)
opt_g_gan = torch.optim.Adam(G_gan.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_d_gan = torch.optim.Adam(D_gan.parameters(), lr=2e-4, betas=(0.5, 0.999))
bce = nn.BCEWithLogitsLoss()
gan_g_losses, gan_d_losses = [], []

for epoch in range(15):
    eg, ed = [], []
    for (real_batch,) in real_loader:
        bs = real_batch.size(0)
        z = torch.randn(bs, LATENT_DIM, device=device)
        fake = G_gan(z).detach()
        loss_d = bce(D_gan(real_batch), torch.ones(bs, 1, device=device)) + bce(
            D_gan(fake), torch.zeros(bs, 1, device=device)
        )
        opt_d_gan.zero_grad()
        loss_d.backward()
        opt_d_gan.step()
        z = torch.randn(bs, LATENT_DIM, device=device)
        loss_g = bce(D_gan(G_gan(z)), torch.ones(bs, 1, device=device))
        opt_g_gan.zero_grad()
        loss_g.backward()
        opt_g_gan.step()
        eg.append(loss_g.item())
        ed.append(loss_d.item())
    gan_g_losses.append(float(np.mean(eg)))
    gan_d_losses.append(float(np.mean(ed)))
    print(
        f"  [Vanilla] epoch {epoch+1:2d}/15  "
        f"D={gan_d_losses[-1]:.3f}  G={gan_g_losses[-1]:.3f}"
    )


# Train WGAN-GP
def gradient_penalty(D, real, fake):
    batch = real.size(0)
    alpha = torch.rand(batch, 1, 1, 1, device=real.device)
    interp = (alpha * real + (1 - alpha) * fake).requires_grad_(True)
    d_interp = D(interp)
    grad = torch.autograd.grad(
        outputs=d_interp,
        inputs=interp,
        grad_outputs=torch.ones_like(d_interp),
        create_graph=True,
        retain_graph=True,
        only_inputs=True,
    )[0]
    return ((grad.reshape(batch, -1).norm(2, dim=1) - 1) ** 2).mean()


print("\n  Training WGAN-GP (20 epochs)...")
G_wgan = Generator().to(device)
D_wgan = Discriminator().to(device)
opt_g_wgan = torch.optim.Adam(G_wgan.parameters(), lr=1e-4, betas=(0.5, 0.9))
opt_d_wgan = torch.optim.Adam(D_wgan.parameters(), lr=1e-4, betas=(0.5, 0.9))
wgan_g_losses, wgan_d_losses = [], []

for epoch in range(20):
    eg, ed = [], []
    for (real_batch,) in real_loader:
        bs = real_batch.size(0)
        for _ in range(5):
            z = torch.randn(bs, LATENT_DIM, device=device)
            fake = G_wgan(z).detach()
            gp = gradient_penalty(D_wgan, real_batch, fake)
            loss_d = D_wgan(fake).mean() - D_wgan(real_batch).mean() + 10.0 * gp
            opt_d_wgan.zero_grad()
            loss_d.backward()
            opt_d_wgan.step()
        z = torch.randn(bs, LATENT_DIM, device=device)
        loss_g = -D_wgan(G_wgan(z)).mean()
        opt_g_wgan.zero_grad()
        loss_g.backward()
        opt_g_wgan.step()
        eg.append(loss_g.item())
        ed.append(loss_d.item())
    wgan_g_losses.append(float(np.mean(eg)))
    wgan_d_losses.append(float(np.mean(ed)))
    print(
        f"  [WGAN-GP] epoch {epoch+1:2d}/20  "
        f"critic={wgan_d_losses[-1]:.3f}  G={wgan_g_losses[-1]:.3f}"
    )



### Checkpoint 2


In [ ]:
assert len(gan_g_losses) == 15, "Vanilla GAN should train 15 epochs"
assert len(wgan_g_losses) == 20, "WGAN-GP should train 20 epochs"
print("\n--- Checkpoint 2 passed --- both generators trained\n")



## Compute FID scores


In [ ]:
print("\n  Computing FID scores...")

N_FID = 10000
G_gan.eval()
G_wgan.eval()

with torch.no_grad():
    gan_fake_01 = (G_gan(torch.randn(N_FID, LATENT_DIM, device=device)) + 1) / 2
    wgan_fake_01 = (G_wgan(torch.randn(N_FID, LATENT_DIM, device=device)) + 1) / 2

rng = np.random.default_rng(42)
real_sub = (X_real[rng.choice(len(X_real), N_FID, replace=False)] + 1) / 2

fid_gan = compute_fid(fid_ext, real_sub, gan_fake_01)
fid_wgan = compute_fid(fid_ext, real_sub, wgan_fake_01)

# Real-vs-real FID floor: two sets of REAL digits (training vs the test
# split no generator has seen). No generator can beat sampling noise.
X_test, y_test = load_mnist_test(device)
X_test_01 = (X_test + 1) / 2
fid_floor = compute_fid(fid_ext, real_sub, X_test_01)

print(f"\n  FID Scores (lower = better; LeNet-64 feature space):")
print(f"    Real vs real (floor): {fid_floor:.3g}")
print(f"    Vanilla GAN: {fid_gan:.2f}  ({fid_gan / fid_floor:.1f}x floor)")
print(f"    WGAN-GP:     {fid_wgan:.2f}  ({fid_wgan / fid_floor:.1f}x floor)")

# Compute mode coverage
cov_gan, dist_gan, ent_gan = mode_coverage(G_gan, fid_ext, device)
cov_wgan, dist_wgan, ent_wgan = mode_coverage(G_wgan, fid_ext, device)

print(f"\n  Mode Coverage (all 10 digit classes):")
print(f"    Vanilla GAN: {cov_gan}/10 classes, entropy={ent_gan:.2f}/3.32")
print(f"    Distribution: {dist_gan}")
print(f"    WGAN-GP:     {cov_wgan}/10 classes, entropy={ent_wgan:.2f}/3.32")
print(f"    Distribution: {dist_wgan}")



### Novelty check: is either generator copying its training images?


Distance from each image to its nearest training image (feature space).


In [ ]:
# For each image, find the distance (in the extractor's feature space) to
# its NEAREST training image. Baseline: the same distance for real test
# digits, which the generators never saw. A memorising generator sits much
# CLOSER to the training set than unseen real digits do (ratio << 1).
N_NOV = 2000


def _features(images_01: torch.Tensor) -> torch.Tensor:
    with torch.no_grad():
        return torch.cat(
            [
                fid_ext.extract_features(images_01[i : i + 5000])
                for i in range(0, len(images_01), 5000)
            ]
        )


train_feats = _features((X_real + 1) / 2)


def nn_distance(images_01: torch.Tensor) -> torch.Tensor:
    q = _features(images_01)
    return torch.cat(
        [
            torch.cdist(q[i : i + 250], train_feats).min(dim=1).values
            for i in range(0, len(q), 250)
        ]
    )


d_unseen = nn_distance(X_test_01[:N_NOV]).median().item()
nov_gan = nn_distance(gan_fake_01[:N_NOV]).median().item() / d_unseen
nov_wgan = nn_distance(wgan_fake_01[:N_NOV]).median().item() / d_unseen

print(f"\n  Novelty (median NN distance to training set / unseen real digits'):")
print(f"    Vanilla GAN: {nov_gan:.2f}")
print(f"    WGAN-GP:     {nov_wgan:.2f}")
print("    ~1.0 = as novel as unseen real digits; well below 1 = near-copies")


# Log to ExperimentTracker
async def _log_evaluation():
    async with tracker.track(experiment=exp_name, run_name="fid_evaluation") as run:
        await run.log_param("n_generated", str(N_FID))
        await run.log_metrics(
            {
                "fid_vanilla_gan": fid_gan,
                "fid_wgan_gp": fid_wgan,
                "coverage_vanilla": float(cov_gan),
                "coverage_wgan": float(cov_wgan),
                "entropy_vanilla": ent_gan,
                "entropy_wgan": ent_wgan,
                "fid_real_floor": fid_floor,
                "novelty_vanilla": nov_gan,
                "novelty_wgan": nov_wgan,
            }
        )


await _log_evaluation()



### Checkpoint 3


In [ ]:
# FID is mathematically non-negative, but the numerical estimate (sample
# covariances + eigenvalues of Sigma_r @ Sigma_g) can land tiny-negative
# from floating-point error. Allow a small tolerance.
assert (
    fid_gan >= -1e-3 and fid_wgan >= -1e-3
), f"FID expected ~0+; got fid_gan={fid_gan:.6f}, fid_wgan={fid_wgan:.6f}"
assert fid_floor >= -1e-3, f"Real-vs-real FID should be ~0+; got {fid_floor:.6f}"
assert nov_gan > 0 and nov_wgan > 0, "Novelty ratios must be positive"
assert 0 <= ent_gan <= np.log2(10) + 0.01, "Entropy out of range"
assert cov_gan >= 1 and cov_wgan >= 1, "Must produce at least 1 class"
# INTERPRETATION: FID = 0 means identical feature distributions. Read
# each FID against the real-vs-real floor printed above, not against
# published Inception-FID numbers — this extractor has its own scale.
# WGAN-GP often achieves better coverage because the critic still gives
# an informative gradient when the distributions barely overlap — but
# check YOUR numbers: one training run is one sample.
print("\n--- Checkpoint 3 passed --- FID and mode coverage computed\n")



## PHASE 4 — VISUALISE: FID Comparison, Mode Coverage, Latent Walk


In [ ]:
print("\n" + "=" * 70)
print("  PHASE 4 — VISUALISE: Evaluation Metrics Dashboard")
print("=" * 70)

# 4A: FID score comparison bar chart
print("\n  4A: FID score comparison")
fig_fid, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(
    ["Vanilla GAN", "WGAN-GP"],
    [fid_gan, fid_wgan],
    color=["#e74c3c", "#2ecc71"],
    width=0.5,
    edgecolor="black",
    linewidth=1.2,
)
ax.set_ylabel("FID Score (lower = better)", fontsize=13)
ax.set_title(
    "Frechet Distance Comparison (LeNet-64 features)\n"
    f"(computed on {N_FID:,} generated vs {N_FID:,} real images)",
    fontsize=14,
    fontweight="bold",
)
# Add value labels on bars
for bar, val in zip(bars, [fid_gan, fid_wgan]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f"{val:.1f}",
        ha="center",
        fontsize=14,
        fontweight="bold",
    )
# Add reference lines
ax.axhline(
    y=fid_floor,
    color="blue",
    linestyle="--",
    alpha=0.6,
    label=f"Real-vs-real floor ({fid_floor:.3g})",
)
ax.legend(fontsize=10, loc="upper right")
ax.grid(True, alpha=0.2, axis="y")
plt.tight_layout()
fig_fid.savefig(
    str(OUTPUT_DIR / "ex_5_03_fid_comparison.png"), dpi=150, bbox_inches="tight"
)
plt.show()

# 4B: Mode coverage matrix — heatmap of generated digit classes
print("\n  4B: Mode coverage matrix (all 10 digit classes)")
fig_mode, axes = plt.subplots(1, 2, figsize=(16, 5))
fig_mode.suptitle(
    "Mode Coverage: Which Digits Does Each GAN Generate?",
    fontsize=14,
    fontweight="bold",
)

for idx, (name, dist, ent, cov) in enumerate(
    [
        ("Vanilla GAN", dist_gan, ent_gan, cov_gan),
        ("WGAN-GP", dist_wgan, ent_wgan, cov_wgan),
    ]
):
    counts = [dist.get(c, 0) for c in range(10)]
    total = sum(counts)
    pcts = [c / total * 100 if total > 0 else 0 for c in counts]
    colors = ["#2ecc71" if c > 0 else "#e74c3c" for c in counts]

    bars = axes[idx].bar(
        range(10), pcts, color=colors, edgecolor="black", linewidth=0.8
    )
    axes[idx].set_xlabel("Digit Class", fontsize=12)
    axes[idx].set_ylabel("% of Generated Images", fontsize=12)
    axes[idx].set_title(
        f"{name}\n{cov}/10 classes, entropy={ent:.2f}/3.32", fontsize=13
    )
    axes[idx].set_xticks(range(10))

    # Ideal uniform line
    axes[idx].axhline(
        y=10, color="blue", linestyle="--", alpha=0.5, label="Ideal uniform (10%)"
    )
    axes[idx].legend(fontsize=10)
    axes[idx].grid(True, alpha=0.2, axis="y")

    # Value labels
    for bar, pct in zip(bars, pcts):
        if pct > 0:
            axes[idx].text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.3,
                f"{pct:.1f}%",
                ha="center",
                fontsize=9,
            )

plt.tight_layout()
fig_mode.savefig(
    str(OUTPUT_DIR / "ex_5_03_mode_coverage.png"), dpi=150, bbox_inches="tight"
)
plt.show()

# 4C: Latent space interpolation walk — smooth transitions
print("\n  4C: Latent space interpolation (WGAN-GP)")
fig_walk = plot_latent_interpolation(
    G_wgan,
    device,
    n_steps=12,
    n_rows=6,
    title="WGAN-GP Latent Walk — Smooth Digit Transitions",
    save_path=str(OUTPUT_DIR / "ex_5_03_latent_walk.png"),
)
plt.show()

# 4D: Combined training dynamics
print("\n  4D: Combined training dynamics comparison")
viz = ModelVisualizer()
fig_all = viz.training_history(
    metrics={
        "Vanilla G": gan_g_losses,
        "Vanilla D": gan_d_losses,
        "WGAN G": wgan_g_losses,
        "WGAN Critic": wgan_d_losses,
    },
    x_label="Epoch",
    y_label="Loss",
)
fig_all.write_html(str(OUTPUT_DIR / "ex_5_03_combined_training.html"))
print("  Interactive combined training curves saved")

# 4E: Comprehensive evaluation dashboard
print("\n  4E: Evaluation dashboard")
fig_dash, axes = plt.subplots(2, 2, figsize=(16, 12))
fig_dash.suptitle(
    "GAN Evaluation Dashboard — Vanilla GAN vs WGAN-GP",
    fontsize=16,
    fontweight="bold",
)

# FID comparison
bars = axes[0][0].bar(
    ["Vanilla GAN", "WGAN-GP"],
    [fid_gan, fid_wgan],
    color=["#e74c3c", "#2ecc71"],
    edgecolor="black",
)
axes[0][0].set_ylabel("FID Score", fontsize=12)
axes[0][0].set_title("Distribution Match (lower = better)", fontsize=13)
for bar, val in zip(bars, [fid_gan, fid_wgan]):
    axes[0][0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.3,
        f"{val:.1f}",
        ha="center",
        fontsize=12,
        fontweight="bold",
    )

# Mode coverage
bars = axes[0][1].bar(
    ["Vanilla GAN", "WGAN-GP"],
    [ent_gan, ent_wgan],
    color=["#e74c3c", "#2ecc71"],
    edgecolor="black",
)
axes[0][1].axhline(
    y=np.log2(10), color="blue", linestyle="--", alpha=0.5, label="Max entropy"
)
axes[0][1].set_ylabel("Shannon Entropy", fontsize=12)
axes[0][1].set_title("Mode Diversity (higher = better)", fontsize=13)
axes[0][1].legend(fontsize=10)
for bar, val in zip(bars, [ent_gan, ent_wgan]):
    axes[0][1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.02,
        f"{val:.2f}",
        ha="center",
        fontsize=12,
        fontweight="bold",
    )

# G losses
axes[1][0].plot(range(1, 16), gan_g_losses, "r-", linewidth=2, label="Vanilla GAN")
axes[1][0].plot(range(1, 21), wgan_g_losses, "g-", linewidth=2, label="WGAN-GP")
axes[1][0].set_xlabel("Epoch", fontsize=12)
axes[1][0].set_ylabel("Generator Loss", fontsize=12)
axes[1][0].set_title("Generator Training Curves", fontsize=13)
axes[1][0].legend(fontsize=11)
axes[1][0].grid(True, alpha=0.3)

# D/Critic losses
axes[1][1].plot(range(1, 16), gan_d_losses, "r-", linewidth=2, label="Vanilla D")
axes[1][1].plot(range(1, 21), wgan_d_losses, "g-", linewidth=2, label="WGAN Critic")
axes[1][1].set_xlabel("Epoch", fontsize=12)
axes[1][1].set_ylabel("Loss", fontsize=12)
axes[1][1].set_title("Discriminator/Critic Curves", fontsize=13)
axes[1][1].legend(fontsize=11)
axes[1][1].grid(True, alpha=0.3)

plt.tight_layout()
fig_dash.savefig(
    str(OUTPUT_DIR / "ex_5_03_evaluation_dashboard.png"), dpi=150, bbox_inches="tight"
)
plt.show()



### Checkpoint 4


In [ ]:
assert os.path.exists(
    str(OUTPUT_DIR / "ex_5_03_fid_comparison.png")
), "FID comparison should exist"
assert os.path.exists(
    str(OUTPUT_DIR / "ex_5_03_evaluation_dashboard.png")
), "Dashboard should exist"
print("\n--- Checkpoint 4 passed --- evaluation visualisations complete\n")



## Register generators in ModelRegistry


In [ ]:
print("\n  Registering generators in ModelRegistry...")
ver_gan = register_generator(
    registry, "vanilla_gan_generator", G_gan, fid_gan, cov_gan, ent_gan
)
ver_wgan = register_generator(
    registry, "wgan_gp_generator", G_wgan, fid_wgan, cov_wgan, ent_wgan
)



### Checkpoint 5


In [ ]:
if registry is not None:
    assert ver_gan is not None, "Vanilla GAN should be registered"
    assert ver_wgan is not None, "WGAN-GP should be registered"
print("\n--- Checkpoint 5 passed --- generators registered\n")



## PHASE 5 — APPLY: QA Pipeline for Insurance Synthetic Data


BUSINESS SCENARIO (illustrative):
  You are the ML engineering lead at the Singapore insurer from
  Exercise 5.1. Your team has trained a WGAN-GP to generate synthetic
  policyholder profiles for fraud detection model training.

  Before deploying synthetic data to production, the Chief Risk Officer
  (CRO) asks: "How do you KNOW the synthetic data is good enough?
  What if the generator produces biased profiles that make the fraud
  model miss certain claim types?"

  THE QA GATE THIS SCRIPT RUNS:
  1. Distribution match: FID within a set multiple of the real-vs-real
     floor (the multiple is a POLICY choice — calibrate it per extractor)
  2. Mode coverage: at least 8 of 10 categories generated
  3. Diversity: Shannon entropy of the generated categories
  4. Balance: no category below 3% of generated samples
  5. Novelty: generated samples must not sit much closer to the
     training set than unseen real samples do (near-copy check)

  GATES THIS SCRIPT DOES NOT RUN (still required before production):
  - Downstream validation: a fraud model trained with the synthetic
    data, scored on REAL held-out claims, against a real-data baseline
  - Privacy review: the novelty check catches near-copies only; it is
    not a privacy guarantee (that needs DP training and membership-
    inference testing)

  Synthetic data that fails any gate is rejected and the generator is
  retrained or tuned.


In [ ]:
print("\n" + "=" * 70)
print("  PHASE 5 — APPLY: Synthetic Data QA for Insurance Production")
print("=" * 70)
print(
)

# Step 1: Define QA thresholds (policy choices, not universal constants)
FID_FLOOR_MULTIPLE = 10.0  # FID at most 10x the real-vs-real floor
FID_THRESHOLD = FID_FLOOR_MULTIPLE * fid_floor
MIN_MODE_COVERAGE = 8  # At least 8/10 classes
MIN_ENTROPY = 2.5  # Minimum diversity (max is 3.32)
MIN_CLASS_PCT = 3.0  # No class below 3% of total
MIN_NOVELTY_RATIO = 0.8  # NN distance at least 80% of unseen real digits'

print("  QA Thresholds:")
print(f"    FID score:        < {FID_THRESHOLD:.3g} ({FID_FLOOR_MULTIPLE:.0f}x floor)")
print(f"    Mode coverage:    >= {MIN_MODE_COVERAGE}/10 classes")
print(f"    Shannon entropy:  >= {MIN_ENTROPY}/3.32")
print(f"    Min class share:  >= {MIN_CLASS_PCT}%")
print(f"    Novelty ratio:    >= {MIN_NOVELTY_RATIO}")

# Step 2: Run QA on both generators
print("\n  Running QA pipeline on both generators...")

qa_results = {}
for name, G, fid, cov, dist, ent, nov in [
    ("Vanilla GAN", G_gan, fid_gan, cov_gan, dist_gan, ent_gan, nov_gan),
    ("WGAN-GP", G_wgan, fid_wgan, cov_wgan, dist_wgan, ent_wgan, nov_wgan),
]:
    # Smallest share over ALL 10 classes — a class never generated counts
    # as 0%, not as "absent from the dict".
    total = sum(dist.values())
    min_pct = min(dist.get(c, 0) for c in range(10)) / total * 100 if total else 0.0

    fid_pass = fid < FID_THRESHOLD
    cov_pass = cov >= MIN_MODE_COVERAGE
    ent_pass = ent >= MIN_ENTROPY
    pct_pass = min_pct >= MIN_CLASS_PCT
    nov_pass = nov >= MIN_NOVELTY_RATIO

    all_pass = fid_pass and cov_pass and ent_pass and pct_pass and nov_pass

    qa_results[name] = {
        "fid": fid,
        "fid_pass": fid_pass,
        "coverage": cov,
        "cov_pass": cov_pass,
        "entropy": ent,
        "ent_pass": ent_pass,
        "min_class_pct": min_pct,
        "pct_pass": pct_pass,
        "novelty": nov,
        "nov_pass": nov_pass,
        "overall": all_pass,
    }

# Step 3: QA results visualisation
fig_qa, ax = plt.subplots(figsize=(14, 7))
fig_qa.suptitle(
    "Synthetic Data QA Pipeline Results\n"
    "Production Gate for Insurance Fraud Model Training",
    fontsize=14,
    fontweight="bold",
)

categories = [
    f"FID Score\n(< {FID_THRESHOLD:.3g})",
    "Mode Coverage\n(>= 8/10)",
    "Entropy\n(>= 2.5)",
    "Min Class %\n(>= 3%)",
    "Novelty Ratio\n(>= 0.8)",
]
vanilla_scores = [
    qa_results["Vanilla GAN"]["fid"],
    qa_results["Vanilla GAN"]["coverage"],
    qa_results["Vanilla GAN"]["entropy"],
    qa_results["Vanilla GAN"]["min_class_pct"],
    qa_results["Vanilla GAN"]["novelty"],
]
wgan_scores = [
    qa_results["WGAN-GP"]["fid"],
    qa_results["WGAN-GP"]["coverage"],
    qa_results["WGAN-GP"]["entropy"],
    qa_results["WGAN-GP"]["min_class_pct"],
    qa_results["WGAN-GP"]["novelty"],
]
thresholds = [
    FID_THRESHOLD,
    MIN_MODE_COVERAGE,
    MIN_ENTROPY,
    MIN_CLASS_PCT,
    MIN_NOVELTY_RATIO,
]

x = np.arange(len(categories))
width = 0.3

# Normalise to threshold (100% = meets threshold)
vanilla_norm = []
wgan_norm = []
for v, w, t, cat in zip(vanilla_scores, wgan_scores, thresholds, categories):
    if "FID" in cat:
        # Lower is better for FID
        vanilla_norm.append(min(t / (v + 1e-8) * 100, 150))
        wgan_norm.append(min(t / (w + 1e-8) * 100, 150))
    else:
        vanilla_norm.append(min(v / t * 100, 150))
        wgan_norm.append(min(w / t * 100, 150))

bars1 = ax.bar(
    x - width / 2, vanilla_norm, width, label="Vanilla GAN", color="#e74c3c", alpha=0.8
)
bars2 = ax.bar(
    x + width / 2, wgan_norm, width, label="WGAN-GP", color="#2ecc71", alpha=0.8
)
ax.axhline(
    y=100, color="black", linestyle="--", linewidth=2, label="QA Threshold (100%)"
)
ax.set_ylabel("% of QA Threshold Met", fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(categories, fontsize=11)
ax.legend(fontsize=11, loc="upper right")
ax.grid(True, alpha=0.2, axis="y")
ax.set_ylim(0, 160)

# Pass/fail indicators
for i, (b1, b2) in enumerate(zip(bars1, bars2)):
    v_pass = [
        qa_results["Vanilla GAN"]["fid_pass"],
        qa_results["Vanilla GAN"]["cov_pass"],
        qa_results["Vanilla GAN"]["ent_pass"],
        qa_results["Vanilla GAN"]["pct_pass"],
        qa_results["Vanilla GAN"]["nov_pass"],
    ][i]
    w_pass = [
        qa_results["WGAN-GP"]["fid_pass"],
        qa_results["WGAN-GP"]["cov_pass"],
        qa_results["WGAN-GP"]["ent_pass"],
        qa_results["WGAN-GP"]["pct_pass"],
        qa_results["WGAN-GP"]["nov_pass"],
    ][i]
    ax.text(
        b1.get_x() + b1.get_width() / 2,
        b1.get_height() + 2,
        "PASS" if v_pass else "FAIL",
        ha="center",
        fontsize=9,
        fontweight="bold",
        color="green" if v_pass else "red",
    )
    ax.text(
        b2.get_x() + b2.get_width() / 2,
        b2.get_height() + 2,
        "PASS" if w_pass else "FAIL",
        ha="center",
        fontsize=9,
        fontweight="bold",
        color="green" if w_pass else "red",
    )

plt.tight_layout()
fig_qa.savefig(
    str(OUTPUT_DIR / "ex_5_03_qa_pipeline.png"), dpi=150, bbox_inches="tight"
)
plt.show()

# Step 4: Stakeholder-ready QA report
van_overall = qa_results["Vanilla GAN"]["overall"]
wgan_overall = qa_results["WGAN-GP"]["overall"]

print("\n  ┌────────────────────────────────────────────────────────────────┐")
print("  │  SYNTHETIC DATA QA REPORT — Insurance Fraud Model Pipeline    │")
print("  ├────────────────────────────────────────────────────────────────┤")
print("  │                                                                │")
print(f"  │  {'Metric':<22} {'Vanilla GAN':>13} {'WGAN-GP':>13} {'Threshold':>12} │")
print(f"  │  {'─'*60}  │")
print(
    f"  │  {'FID Score':<22} {fid_gan:>13.1f} {fid_wgan:>13.1f} "
    f"{'< ' + format(FID_THRESHOLD, '.3g'):>12} │"
)
print(
    f"  │  {'Mode Coverage':<22} {cov_gan:>12}/10 {cov_wgan:>12}/10 {'>= 8/10':>12} │"
)
print(
    f"  │  {'Shannon Entropy':<22} {ent_gan:>13.2f} {ent_wgan:>13.2f} {'>= 2.50':>12} │"
)
van_min = qa_results["Vanilla GAN"]["min_class_pct"]
wgan_min = qa_results["WGAN-GP"]["min_class_pct"]
print(
    f"  │  {'Min Class Share':<22} {van_min:>12.1f}% {wgan_min:>12.1f}% {'>= 3.0%':>12} │"
)
print(
    f"  │  {'Novelty Ratio':<22} {nov_gan:>13.2f} {nov_wgan:>13.2f} {'>= 0.80':>12} │"
)
print("  │                                                                │")
van_status = "APPROVED" if van_overall else "REJECTED"
wgan_status = "APPROVED" if wgan_overall else "REJECTED"
print(f"  │  {'OVERALL STATUS':<22} {van_status:>13} {wgan_status:>13}              │")
print("  │                                                                │")
better = "WGAN-GP" if fid_wgan < fid_gan else "Vanilla GAN"
print(f"  │  Lower FID: {better:<50} │")
print(f"  │  Best FID score: {min(fid_gan, fid_wgan):<44.1f} │")
print("  │                                                                │")
approved = [n for n, r in qa_results.items() if r["overall"]]
if approved:
    decision = f"{', '.join(approved)} passed this gate"
    follow_up = "Next: downstream test on REAL claims + privacy review"
else:
    decision = "No generator passed — retrain or tune before reuse"
    follow_up = "See the FAIL bars above for which gate blocked each one"
print(f"  │  DECISION: {decision:<51} │")
print(f"  │  {follow_up:<61} │")
print("  └────────────────────────────────────────────────────────────────┘")



### Checkpoint 6


In [ ]:
assert os.path.exists(
    str(OUTPUT_DIR / "ex_5_03_qa_pipeline.png")
), "QA pipeline chart should exist"
print("\n--- Checkpoint 6 passed --- QA pipeline complete\n")



## Final Summary


In [ ]:
print("\n" + "=" * 70)
print("  EXPERIMENT SUMMARY")
print("=" * 70)
print(f"\n  Experiment: {exp_name}")
print(f"  Dataset: MNIST (60,000 images), latent_dim={LATENT_DIM}")
print(f"\n  {'Metric':<25} {'Vanilla GAN':>14} {'WGAN-GP':>14}")
print(f"  {'-'*53}")
print(f"  {'Epochs':<25} {'15':>14} {'20':>14}")
print(f"  {'Final G loss':<25} {gan_g_losses[-1]:>14.3f} {wgan_g_losses[-1]:>14.3f}")
print(
    f"  {'Final D/Critic loss':<25} {gan_d_losses[-1]:>14.3f} {wgan_d_losses[-1]:>14.3f}"
)
print(f"  {'FID score':<25} {fid_gan:>14.2f} {fid_wgan:>14.2f}")
print(f"  {'Mode coverage':<25} {cov_gan:>13}/10 {cov_wgan:>13}/10")
print(f"  {'Class entropy':<25} {ent_gan:>14.2f} {ent_wgan:>14.2f}")
print(f"  {'Novelty ratio':<25} {nov_gan:>14.2f} {nov_wgan:>14.2f}")
print(f"\n  Best generator by FID: {better}")



## Cleanup


In [ ]:
await close_engines(conn)



## DIAGNOSTIC CHECKPOINT — WGAN-GP generator, one kailash-ml call


In [ ]:
# FID, coverage and novelty judge the OUTPUT. The Prescription Pad judges
# the network: run_diagnostic_checkpoint instruments the trained WGAN-GP
# generator, replays a few batches of its real training objective (score
# fakes with the trained critic; no weights are updated) and returns the
# gradient-flow, dead-neuron and loss-trend readings.
from kailash_ml.diagnostics import run_diagnostic_checkpoint


def _wgan_g_loss(m, batch):
    z = torch.randn(batch[0].size(0), LATENT_DIM, device=device)
    return -D_wgan(m(z)).mean()


diag, findings = run_diagnostic_checkpoint(
    G_wgan,
    real_loader,
    _wgan_g_loss,
    title="WGAN-GP generator",
    n_batches=8,
    train_losses=wgan_g_losses,
    show=False,
)
print_prescription_pad(findings, "WGAN-GP generator")
# HOW TO READ IT: put this pad beside the QA report. Healthy gradients
# with a FAILED coverage gate means the network trains fine but has
# collapsed onto a few modes — a data/objective problem, not a plumbing
# one. Vanishing gradients point at the critic (under-trained, or GP not
# applied). For a GAN, a loss-trend WARNING is not by itself a failure:
# G's loss moves as the critic gets stronger.



## REFLECTION


GAN EVALUATION METRICS:
  [x] FID (Frechet Inception Distance): the standard automated GAN
      metric. Measures distributional distance in a learned feature
      space — only comparable within the same extractor, so read it
      against a real-vs-real floor.
  [x] Mode coverage: counting how many classes the generator produces.
      A sharp but repetitive GAN fails this test.
  [x] Shannon entropy: quantifying generation diversity.
      Max = log2(10) = 3.32 for MNIST (uniform across all digits).
  [x] Minimum class share: no category can be underrepresented
      (prevents hidden bias in synthetic datasets).
  [x] Novelty ratio: a near-copy check against the training set
      (necessary for privacy, nowhere near sufficient).

  MODEL REGISTRY:
  [x] Registered both generators with FID + coverage + entropy metrics
  [x] Registered metrics let you compare generator versions side by side

  QA PIPELINE FOR PRODUCTION:
  [x] Defined quantitative thresholds (FID vs floor, coverage >= 8/10, ...)
  [x] Automated evaluation — no "looks good to me" subjectivity
  [x] Stakeholder-ready report whose decision is computed, not asserted
  [x] Named the gates this script does NOT run (downstream test, privacy)

  REAL-WORLD APPLICATION:
  [x] Insurance synthetic data QA: evidence before production
  [x] CRO-ready quality report with pass/fail per metric
  [x] The business risk of biased or memorised synthetic data

  KEY INSIGHTS:
  - FID alone is not enough: it blends fidelity and diversity into one
    number, so it cannot say WHICH of the two is failing
  - Mode coverage alone is not enough: a generator can cover all modes
    but produce blurry, low-quality images
  - You need BOTH distribution match (FID) AND diversity (coverage +
    entropy) — and a novelty check before anyone calls the data safe
  - Re-run the same gate on every new generator version; the registry
    keeps the metrics side by side so regressions are visible

  WHEN TO USE WHICH GAN:
  - Vanilla GAN: quick prototyping, simple datasets, no stability needs
  - WGAN-GP: production use, medical/financial data, stability required
  - Both need the same evaluation pipeline — the QA doesn't change,
    only the generator architecture does.

  GAN vs VAE (M5 Exercise 1) vs Diffusion:
  - GANs:      Sharp images, hard to train, fast sampling
  - VAEs:      Blurry but stable, continuous latent space, fast
  - Diffusion: Sharp + stable, best quality, SLOW sampling


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

